In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# =========================
# CONFIG
# =========================
INPUT_URL_FILE = r"C:\Users\mdumiseni\Documents\data science assignements\data-science-portfolio\Nutrition_risk_african_recipes\allrecipes_urls.txt"
OUTPUT_CSV = "african_recipes_dataset.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

# =========================
# COOKING METHOD KEYWORDS
# =========================

COOKING_METHODS = [
    "boil",
    "bake",
    "fry",
    "deep fry",
    "steam",
    "grill",
    "roast",
    "simmer",
    "stew",
    "poach",
    "braise",
    "smoke",
    "toast",
    "saute",
    "sauté",
    "microwave",
    "pressure cook",
    "slow cook",
    "blend",
    "marinate",
    "broil"
]

# =========================
# NORMALIZE FRACTIONS
# =========================

FRACTIONS = {
    "¼": "1/4",
    "½": "1/2",
    "¾": "3/4",
    "⅓": "1/3",
    "⅔": "2/3",
    "⅛": "1/8",
    "⅜": "3/8",
    "⅝": "5/8",
    "⅞": "7/8"
}

def normalize_fractions(text):
    for k, v in FRACTIONS.items():
        text = text.replace(k, v)
    return text

# =========================
# EXTRACT COOKING METHODS
# =========================

def extract_cooking_methods(text):
    text = text.lower()

    found = []

    for method in COOKING_METHODS:
        if method in text:
            found.append(method)

    return ", ".join(sorted(set(found)))

# =========================
# SCRAPE RECIPE
# =========================

def scrape_recipe(url):

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)

        if response.status_code != 200:
            print(f"Failed: {url}")
            return None

        soup = BeautifulSoup(response.text, "lxml")

        # =========================
        # TITLE
        # =========================

        title = ""

        title_tag = soup.find("h1")

        if title_tag:
            title = title_tag.get_text(" ", strip=True)

        # =========================
        # SERVINGS
        # =========================

        servings = ""

        servings_tag = soup.find(
            attrs={"data-ingredient-serving-size": True}
        )

        if servings_tag:
            servings = servings_tag.get_text(strip=True)

        if not servings:
            serving_text = soup.get_text(" ", strip=True)

            serving_match = re.search(
                r"(\d+)\s+servings?",
                serving_text,
                re.IGNORECASE
            )

            if serving_match:
                servings = serving_match.group(1)

        # =========================
        # INGREDIENTS
        # =========================

        ingredients = []

        ingredient_items = soup.select(
            "li.mm-recipes-structured-ingredients__list-item"
        )

        for item in ingredient_items:

            qty = item.select_one(
                "[data-ingredient-quantity='true']"
            )

            unit = item.select_one(
                "[data-ingredient-unit='true']"
            )

            name = item.select_one(
                "[data-ingredient-name='true']"
            )

            qty_text = qty.get_text(" ", strip=True) if qty else ""
            unit_text = unit.get_text(" ", strip=True) if unit else ""
            name_text = name.get_text(" ", strip=True) if name else ""

            ingredient = f"{qty_text} {unit_text} {name_text}".strip()

            ingredient = normalize_fractions(ingredient)

            ingredients.append(ingredient)

        ingredients_text = " | ".join(ingredients)

        # =========================
        # DIRECTIONS
        # =========================

        directions = []

        direction_tags = soup.select(
            "div.mm-recipes-steps__content p"
        )

        for step in direction_tags:
            step_text = step.get_text(" ", strip=True)

            if len(step_text) > 20:
                directions.append(step_text)

        directions_text = " ".join(directions)

        # =========================
        # COOKING METHODS
        # =========================

        cooking_methods = extract_cooking_methods(
            directions_text
        )

        # =========================
        # RETURN DATA
        # =========================

        return {
            "recipe_title": title,
            "servings": servings,
            "ingredients": ingredients_text,
            "cooking_methods": cooking_methods,
            "source_url": url
        }

    except Exception as e:
        print(f"Error scraping {url}")
        print(e)
        return None

# =========================
# LOAD URLS
# =========================

with open(INPUT_URL_FILE, "r", encoding="utf-8") as f:
    urls = [line.strip() for line in f if line.strip()]

# =========================
# SCRAPE ALL RECIPES
# =========================

all_data = []

for i, url in enumerate(urls, start=1):

    print(f"[{i}/{len(urls)}] Scraping: {url}")

    data = scrape_recipe(url)

    if data:
        all_data.append(data)

    time.sleep(2)

# =========================
# EXPORT CSV
# =========================

df = pd.DataFrame(all_data)

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nDONE")
print(f"Recipes scraped: {len(df)}")
print(f"Saved to: {OUTPUT_CSV}")

[1/59] Scraping: https://www.allrecipes.com/recipe/104444/cape-malay-pickled-fish/
[2/59] Scraping: https://www.allrecipes.com/recipe/105044/lamb-tagine/
[3/59] Scraping: https://www.allrecipes.com/recipe/105045/moroccan-couscous/
[4/59] Scraping: https://www.allrecipes.com/recipe/12960/moroccan-lentil-soup/
[5/59] Scraping: https://www.allrecipes.com/recipe/12978/egusi-soup/
[6/59] Scraping: https://www.allrecipes.com/recipe/137967/african-style-oxtail-stew/
[7/59] Scraping: https://www.allrecipes.com/recipe/13988/marrakesh-vegetable-curry/
[8/59] Scraping: https://www.allrecipes.com/recipe/150574/ras-el-hanout/
[9/59] Scraping: https://www.allrecipes.com/recipe/152937/ethiopian-cabbage-dish/
[10/59] Scraping: https://www.allrecipes.com/recipe/157881/best-bobotie/
[11/59] Scraping: https://www.allrecipes.com/recipe/16738/milk-tart/
[12/59] Scraping: https://www.allrecipes.com/recipe/173422/egyptian-koshary/
[13/59] Scraping: https://www.allrecipes.com/recipe/18182/moroccan-chicken/
[1